# exp141_z_driven_pf_z_candidate_gate train

`likpf_mean` を default に固定し、Z-driven 区間だけ `pf_z` を低頻度に選ぶ posthoc gate audit。

## Contents

1. Setup and configuration
2. Input and gate plan
3. Run audit
4. Metrics and artifacts

## 1. Setup and configuration


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

from settings import EXPERIMENT_NAME, ExperimentPaths, load_config
from z_driven_pf_z_candidate_gate import EXP072_FEATURES, find_input_file, run_train_from_config

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()

print('Experiment:', EXPERIMENT_NAME)
print('Route:', config['experiment']['route'])
print('Parent:', config['lineage']['parent'])
print('Output root:', paths.output_root)
print('Artifacts:', paths.artifacts_dir)
print('Gate variants:', len(config['audit']['gate_variants']))

## 2. Input and gate plan


In [ ]:
feature_path = find_input_file(
    EXP072_FEATURES,
    config['data']['exp072_feature_cache'],
    local_roots=[Path('/tmp/exp072_cache_redownload'), Path('/tmp/kaggle-output/exp072_exp063_full_replay_feature_cache/train_v1')],
)
print('Feature cache:', feature_path)
print('Feature cache size:', feature_path.stat().st_size)

gate_plan = pd.DataFrame(config['audit']['gate_variants'])
display(gate_plan[['name', 'scope', 'switch_rate_cap', 'min_conditions', 'min_tail_rank', 'min_md_since', 'alpha', 'clip_abs']])

## 3. Run audit


In [ ]:
summary = run_train_from_config(config, output_dir=paths.artifacts_dir)
print(json.dumps({
    'rows': summary['rows'],
    'wells': summary['wells'],
    'base': summary['base'],
    'best': summary['best'],
    'oracle': summary['oracle'],
}, indent=2))

## 4. Metrics and artifacts


In [ ]:
metrics_path = paths.artifacts_dir / 'exp141_z_driven_pf_z_candidate_gate_metrics.csv'
bucket_path = paths.artifacts_dir / 'exp141_z_driven_pf_z_candidate_gate_bucket_metrics.csv'
parity_path = paths.artifacts_dir / 'exp141_z_driven_pf_z_candidate_gate_rawtest_parity_checklist.csv'

metrics = pd.read_csv(metrics_path)
display(metrics.sort_values('rmse').head(12))

bucket_metrics = pd.read_csv(bucket_path)
display(bucket_metrics.sort_values('base_likpf_mean_rmse', ascending=False).head(12))

parity = pd.read_csv(parity_path)
display(parity)

print('Artifacts:')
for path in sorted(paths.artifacts_dir.glob('exp141_z_driven_pf_z_candidate_gate*')):
    print('-', path.name, path.stat().st_size)